In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# Fake News Detection — Binary Text Classification

## 1. Project Overview

**Task:** Binary text classification — determine whether a news article is **real** or **fake** using only the article text.

**Dataset:** ~20,000 news articles with binary labels (0 = reliable, 1 = unreliable/fake). The dataset mixes political news articles from various US sources (2015–2018).

**Approach:**
1. **Baseline:** TF-IDF vectorization + classical ML (Naive Bayes, Logistic Regression)
2. **Main model:** ModernBERT transformer fine-tuning on local GPU with mixed-precision (fp16)

> **Why not LLMs?** This is a supervised classification task with labeled data. A fine-tuned classifier is faster, cheaper, and more accurate than prompting an LLM. LLMs are better for zero-shot or generative tasks.

---

## 2. Learning Objectives

| # | Skill |
|---|-------|
| 1 | Load and validate a real-world fake news dataset |
| 2 | Build TF-IDF + classical ML baselines for text classification |
| 3 | Fine-tune ModernBERT for binary classification on local GPU |
| 4 | Compare baselines vs transformer with proper metrics |
| 5 | Interpret confusion matrices, precision/recall tradeoffs, and ROC-AUC |
| 6 | Analyze classification errors to understand model limitations |
| 7 | Understand why fake news detection is fundamentally hard |

## 3. Problem Statement

Misinformation spreads faster than corrections on social media. Manually fact-checking every article is impossible at scale. We need automated classifiers that can flag potentially unreliable articles for human reviewers.

**Goal:** Build a classifier that reads a news article and predicts whether it is **reliable (0)** or **unreliable/fake (1)**.

This is a **binary text classification** problem with:
- **Two classes:** real vs fake
- **Long-form text:** full news articles (hundreds to thousands of words)
- **Noisy labels:** not all "fake" labels are clearly fabricated — some are partisan or misleading rather than outright false

## 4. Why This Project Matters

- **Critical real-world problem:** Misinformation affects elections, public health, and financial markets
- **Classic NLP task:** Text classification is the most deployed NLP application in industry
- **Baseline-first mindset:** Shows that TF-IDF + Logistic Regression is often surprisingly competitive
- **Ethical complexity:** Demonstrates that "fake news detection" is more nuanced than a simple binary label
- **Engineering practice:** Proper evaluation, error analysis, and understanding model limitations

## 5. Prerequisites & Assumptions

**Knowledge you should have:**
- Basic Python (functions, loops, f-strings)
- Familiarity with pandas DataFrames
- Understanding of train/test splits and classification concepts
- Basic understanding of what precision and recall mean

**Environment assumptions:**
- Python 3.10+ (3.11 or 3.13 recommended)
- NVIDIA GPU with CUDA support (required for the ModernBERT section; baseline models run on CPU)
- ~4 GB disk space for dataset + model weights
- Internet access for the initial dataset download from HuggingFace

**What you DON'T need:**
- No cloud API keys — everything runs locally
- No prior NLP or deep learning experience — concepts are explained as we go

## 6. Dataset Overview

We use the **Fake News** dataset originally from a Kaggle competition, available on HuggingFace as `GonzaloA/fake_news`.

| Column | Description | Used? |
|--------|-------------|-------|
| `title` | Article headline | ❌ Not used (could be added as a feature) |
| `text` | Full article body | ✅ Input text |
| `label` | 0 = reliable, 1 = unreliable/fake | ✅ Target |

### Source & License

| Property | Value |
|----------|-------|
| **Source** | [Kaggle Fake News Competition](https://www.kaggle.com/c/fake-news) |
| **HuggingFace** | `GonzaloA/fake_news` |
| **License** | Public domain / CC0 (competition dataset) |
| **Size** | ~20,000 articles after filtering |

### Known Data Quality Issues

1. **Labeling ambiguity:** "Fake" includes propaganda, satire, and misleading content — not just outright fabrications
2. **Political bias:** Dataset is US-centric with heavy political content from 2015-2018
3. **Missing text:** Some rows have NaN text or very short text
4. **Temporal bias:** Models trained on this data may not generalize to current news
5. **Author as signal:** Author name alone can be predictive — a sign of dataset bias, not genuine detection

## 7. Environment Setup

The following cell verifies that required packages are available. Uncomment the `pip install` line only if something is missing.

In [ ]:
# Install dependencies (skip if already installed)
# !pip install -q datasets scikit-learn matplotlib seaborn transformers accelerate torch
print("Dependencies: datasets, scikit-learn, matplotlib, seaborn, transformers, torch")
print("(Uncomment the pip install line above if any are missing)")

## 8. Imports

In [ ]:
import os
import time
import warnings
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix, roc_auc_score, roc_curve,
)

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100

%matplotlib inline
print("All imports loaded successfully")

## 9. Configuration

All tuneable parameters in one place. Change sample sizes or model name here — the rest of the notebook adapts automatically.

In [ ]:
# ── Data ──────────────────────────────────────────────────
MAX_SAMPLES = 20_000         # Max rows to use (caps runtime for transformer)
TEST_SIZE = 0.2
RANDOM_STATE = 42

# ── TF-IDF ────────────────────────────────────────────────
TFIDF_MAX_FEATURES = 30_000
TFIDF_NGRAM_RANGE = (1, 2)   # Unigrams + bigrams

# ── Cross-Validation ─────────────────────────────────────
CV_FOLDS = 5

# ── ModernBERT (Advanced Section) ─────────────────────────
BERT_MODEL_NAME = "answerdotai/ModernBERT-base"
BERT_MAX_LEN = 256
BERT_BATCH_SIZE = 16
BERT_EPOCHS = 3
BERT_LR = 2e-5

print("Configuration loaded")

## 10. Dataset Loading

We load the dataset from HuggingFace. The dataset contains news articles with binary labels. We keep only the `text` and `label` columns, dropping rows with missing or very short text.

In [ ]:
from datasets import load_dataset

# ── Load from HuggingFace ────────────────────────────────
print("Loading dataset from HuggingFace...")
ds = load_dataset("GonzaloA/fake_news", split="train", trust_remote_code=True)
df_raw = ds.to_pandas()
print(f"Raw dataset: {len(df_raw):,} rows, {len(df_raw.columns)} columns")
print(f"Columns: {list(df_raw.columns)}")

# ── Select and clean columns ─────────────────────────────
# Auto-detect text column
TEXT_COL = "text"
TARGET = "label"

if TEXT_COL not in df_raw.columns:
    candidates = [c for c in df_raw.columns if df_raw[c].dtype == "object" and df_raw[c].str.len().mean() > 20]
    TEXT_COL = candidates[0] if candidates else df_raw.select_dtypes("object").columns[0]

if TARGET not in df_raw.columns:
    TARGET = df_raw.columns[-1]

df = df_raw[[TEXT_COL, TARGET]].copy()
df.columns = ["text", "label"]

# ── Filter: drop NaN and very short texts ────────────────
before = len(df)
df = df.dropna(subset=["text"])
df = df[df["text"].astype(str).str.strip().str.len() > 50]
df = df.reset_index(drop=True)
print(f"Filtered: {before:,} → {len(df):,} rows (removed {before - len(df):,} without usable text)")

# ── Cap sample size for reasonable runtime ────────────────
if len(df) > MAX_SAMPLES:
    df = df.sample(n=MAX_SAMPLES, random_state=RANDOM_STATE).reset_index(drop=True)
    print(f"Sampled down to {len(df):,} rows for manageable runtime")

print(f"\nFinal dataset: {len(df):,} rows")

## 11. Data Validation

Defensive checks before proceeding: missing values, duplicates, label distribution, and text quality.

In [ ]:
# ── Basic quality checks ──────────────────────────────────
print("Shape:", df.shape)
print("\nMissing values:")
print(df.isnull().sum())

n_dup_rows = df.duplicated().sum()
n_dup_text = df["text"].duplicated().sum()
print(f"\nDuplicate rows: {n_dup_rows:,}")
print(f"Duplicate texts: {n_dup_text:,}")

# ── Remove exact duplicate rows ──────────────────────────
if n_dup_rows > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"  → Removed {n_dup_rows:,} duplicate rows. New shape: {df.shape}")

# ── Label distribution ───────────────────────────────────
print(f"\nLabel distribution:")
for label, count in df["label"].value_counts().sort_index().items():
    label_name = "Reliable" if label == 0 else "Fake/Unreliable"
    print(f"  {label} ({label_name}): {count:,} ({count/len(df):.1%})")

# ── Verify labels are binary ─────────────────────────────
unique_labels = sorted(df["label"].unique())
assert set(unique_labels).issubset({0, 1}), f"Expected binary labels, got: {unique_labels}"
print(f"\n✓ Labels are binary: {unique_labels}")

## 12. Exploratory Data Analysis

### 12a. Class Distribution

Understanding the balance between real and fake articles helps interpret model performance and decide whether class weighting is needed.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
vc = df["label"].value_counts().sort_index()
labels = ["Reliable (0)", "Fake (1)"]
colors = ["steelblue", "salmon"]
bars = ax.bar(labels, vc.values, color=colors, edgecolor="black", alpha=0.85)
for bar, count in zip(bars, vc.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + len(df)*0.005,
            f"{count:,} ({count/len(df):.1%})", ha="center", fontsize=11)
ax.set_title("Class Distribution: Real vs Fake", fontsize=14)
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

### 12b. Text Length Analysis

Understanding text length distribution helps choose model parameters (TF-IDF max features, transformer max sequence length) and anticipate truncation effects.

In [ ]:
df["text_len"] = df["text"].astype(str).str.len()
df["word_count"] = df["text"].astype(str).str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Character length distribution
axes[0].hist(df["text_len"].clip(upper=df["text_len"].quantile(0.99)),
             bins=60, color="steelblue", edgecolor="white", alpha=0.8)
axes[0].set_title("Text Length Distribution (characters)")
axes[0].set_xlabel("Characters")
axes[0].axvline(df["text_len"].median(), color="red", linestyle="--",
                label=f"Median: {df['text_len'].median():.0f}")
axes[0].legend()

# Word count distribution
axes[1].hist(df["word_count"].clip(upper=df["word_count"].quantile(0.99)),
             bins=60, color="darkorange", edgecolor="white", alpha=0.8)
axes[1].set_title("Word Count Distribution")
axes[1].set_xlabel("Words")
axes[1].axvline(df["word_count"].median(), color="red", linestyle="--",
                label=f"Median: {df['word_count'].median():.0f}")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Text length — mean: {df['text_len'].mean():.0f}, median: {df['text_len'].median():.0f}, "
      f"min: {df['text_len'].min()}, max: {df['text_len'].max()}")
print(f"Word count  — mean: {df['word_count'].mean():.0f}, median: {df['word_count'].median():.0f}")

# Clean up temporary columns
df = df.drop(columns=["text_len", "word_count"])

### 12c. Sample Articles

Let's look at a few real and fake articles to understand the data qualitatively.

In [ ]:
for label_val, label_name in [(0, "RELIABLE"), (1, "FAKE")]:
    subset = df[df["label"] == label_val]
    print(f"\n{'='*60}")
    print(f" {label_name} — Sample articles")
    print(f"{'='*60}")
    for i, (_, row) in enumerate(subset.head(2).iterrows()):
        text = str(row["text"])
        preview = text[:300] + "..." if len(text) > 300 else text
        print(f"\n[{i+1}] {preview}")
        print()

## 13. Data Preprocessing

For TF-IDF, minimal preprocessing is usually best — the vectorizer handles lowercasing and stop word removal internally. We just ensure consistent formatting and encode the target.

> **Why minimal preprocessing?** TF-IDF with stop word removal already handles most noise. Over-cleaning (heavy stemming, lemmatizing) can actually hurt performance by destroying useful signal. For a fake news classifier, specific word choices and writing style are informative features.

In [ ]:
# ── Train/test split (stratified) ─────────────────────────
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["text"], df["label"],
    test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=df["label"],
)

print(f"Train: {len(X_train_text):,} samples")
print(f"Test:  {len(X_test_text):,} samples")
print(f"\nTrain label distribution:")
for label, count in y_train.value_counts().sort_index().items():
    print(f"  {label}: {count:,} ({count/len(y_train):.1%})")

## 14. Feature Engineering — TF-IDF Vectorization

**TF-IDF** (Term Frequency — Inverse Document Frequency) converts text to numerical vectors:
- **TF:** How often a term appears in a document (more = more important for that doc)
- **IDF:** How rare the term is across all documents (rarer = more discriminative)

We use unigrams + bigrams to capture both single words and two-word phrases.

> **Important:** We `fit` the vectorizer on training data only, then `transform` both train and test. This prevents data leakage from the test set vocabulary.

In [ ]:
tfidf = TfidfVectorizer(
    max_features=TFIDF_MAX_FEATURES,
    ngram_range=TFIDF_NGRAM_RANGE,
    stop_words="english",
    sublinear_tf=True,       # Use log(1 + tf) instead of raw counts
    min_df=5,                # Ignore terms in < 5 documents
    norm="l2",               # L2 normalize each row
)

X_train_tfidf = tfidf.fit_transform(X_train_text)
X_test_tfidf = tfidf.transform(X_test_text)

print(f"TF-IDF matrix shape: {X_train_tfidf.shape}")
print(f"  {X_train_tfidf.shape[0]:,} documents × {X_train_tfidf.shape[1]:,} features")
print(f"  Sparsity: {1 - X_train_tfidf.nnz / (X_train_tfidf.shape[0] * X_train_tfidf.shape[1]):.4%}")

## 15. Baseline Models — TF-IDF + Classical Classifiers

We compare two classical classifiers using cross-validation:

| Model | Strengths | Weaknesses |
|-------|-----------|------------|
| **MultinomialNB** | Very fast, good text baseline | Assumes feature independence |
| **LogisticRegression** | Strong, interpretable linear model | Slower on very high-dim data |

> **Why cross-validation?** A single train/test split can be misleading — CV gives us a more robust performance estimate with confidence intervals.

In [ ]:
baseline_models = [
    ("Naive Bayes", MultinomialNB()),
    ("Logistic Regression", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, n_jobs=-1)),
]

cv_results = []
for name, clf in baseline_models:
    print(f"  Cross-validating {name}...", end=" ")
    try:
        import mlflow as _mlflow_cv; _mlflow_cv.sklearn.autolog(disable=True)
    except Exception:
        pass
    scores = cross_val_score(clf, X_train_tfidf, y_train, cv=CV_FOLDS,
                             scoring="accuracy", n_jobs=-1)
    for fold, score in enumerate(scores):
        cv_results.append({"model": name, "fold": fold, "accuracy": score})
    print(f"mean={scores.mean():.4f} ± {scores.std():.4f}")

cv_df = pd.DataFrame(cv_results)

# ── Visualization ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=cv_df, x="model", y="accuracy", ax=ax, palette="Set2")
sns.stripplot(data=cv_df, x="model", y="accuracy", ax=ax,
              size=8, jitter=True, edgecolor="gray", linewidth=1.5, alpha=0.7)
ax.set_title(f"Baseline Comparison ({CV_FOLDS}-Fold Cross-Validation)", fontsize=14)
ax.set_ylabel("Accuracy")
ax.set_xlabel("")
plt.tight_layout()
plt.show()

summary = cv_df.groupby("model")["accuracy"].agg(["mean", "std"]).sort_values("mean", ascending=False)
summary.columns = ["Mean Accuracy", "Std"]
print("\n" + summary.to_string())

## 16. Best Baseline — Training & Evaluation

We train **Logistic Regression** (typically the stronger baseline) on the full training set and evaluate on the held-out test set with comprehensive metrics.

In [ ]:
# ── Train best baseline ───────────────────────────────────
t0 = time.perf_counter()
best_baseline = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, n_jobs=-1)
best_baseline.fit(X_train_tfidf, y_train)
baseline_time = time.perf_counter() - t0

# ── Predict ──────────────────────────────────────────────
y_pred_baseline = best_baseline.predict(X_test_tfidf)
y_prob_baseline = best_baseline.predict_proba(X_test_tfidf)[:, 1]

# ── Metrics ──────────────────────────────────────────────
bl_acc = accuracy_score(y_test, y_pred_baseline)
bl_f1 = f1_score(y_test, y_pred_baseline)
bl_prec = precision_score(y_test, y_pred_baseline)
bl_rec = recall_score(y_test, y_pred_baseline)
bl_auc = roc_auc_score(y_test, y_prob_baseline)

print(f"Logistic Regression (TF-IDF) Results:  ({baseline_time:.1f}s)")
print(f"  Accuracy:  {bl_acc:.4f}")
print(f"  Precision: {bl_prec:.4f}")
print(f"  Recall:    {bl_rec:.4f}")
print(f"  F1:        {bl_f1:.4f}")
print(f"  ROC-AUC:   {bl_auc:.4f}")
print()
print(classification_report(y_test, y_pred_baseline,
      target_names=["Reliable", "Fake"], zero_division=0))

### 16b. Confusion Matrix

In [ ]:
cm_baseline = confusion_matrix(y_test, y_pred_baseline)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm_baseline, annot=True, fmt="d", cmap="Blues", ax=ax,
            xticklabels=["Reliable", "Fake"], yticklabels=["Reliable", "Fake"])
ax.set_ylabel("Actual", fontsize=12)
ax.set_xlabel("Predicted", fontsize=12)
ax.set_title("Logistic Regression — Confusion Matrix", fontsize=14)
plt.tight_layout()
plt.show()

## 17. Main Model — ModernBERT Fine-Tuning (GPU)

> **This section requires a CUDA-capable GPU.** It fine-tunes a transformer encoder for binary classification — significantly more expensive than TF-IDF but can capture deeper semantic patterns.

**ModernBERT** (`answerdotai/ModernBERT-base`) is a 2024 English-focused encoder that improves on BERT and RoBERTa with better training recipes and architecture tweaks.

We use:
- Mixed-precision (fp16) for faster training
- Learning rate warmup + linear decay
- Gradient clipping for stability
- A capped data subset to keep training time reasonable (~3–8 min on RTX 4060)

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ── Prepare data for transformer ─────────────────────────
# Use indices from the same train/test split to ensure fair comparison
bert_train_text = X_train_text.tolist()
bert_test_text = X_test_text.tolist()
bert_train_labels = y_train.tolist()
bert_test_labels = y_test.tolist()

print(f"\nModernBERT data — Train: {len(bert_train_text):,}, Test: {len(bert_test_text):,}")

### 17b. Tokenization & DataLoaders

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    BERT_MODEL_NAME, num_labels=2
).to(device)

class NewsDataset(Dataset):
    def __init__(self, texts, labels):
        self.enc = tokenizer(
            texts, truncation=True, padding="max_length",
            max_length=BERT_MAX_LEN, return_tensors="pt",
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        return {**{k: v[i] for k, v in self.enc.items()}, "labels": self.labels[i]}

train_loader = DataLoader(
    NewsDataset(bert_train_text, bert_train_labels),
    batch_size=BERT_BATCH_SIZE, shuffle=True,
)
test_loader = DataLoader(
    NewsDataset(bert_test_text, bert_test_labels),
    batch_size=BERT_BATCH_SIZE,
)
print(f"DataLoaders ready — {len(train_loader)} train batches, {len(test_loader)} test batches")

## 18. Training

Fine-tune with AdamW optimizer, linear warmup scheduling, and mixed-precision (fp16) on GPU.

In [ ]:
use_amp = device.type == "cuda"
optimizer = torch.optim.AdamW(model.parameters(), lr=BERT_LR, weight_decay=0.01)
total_steps = len(train_loader) * BERT_EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

t0 = time.perf_counter()
for epoch in range(BERT_EPOCHS):
    model.train()
    total_loss = 0
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.amp.autocast("cuda", enabled=use_amp):
            loss = model(**batch).loss
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        optimizer.zero_grad()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)
    elapsed = time.perf_counter() - t0
    print(f"  Epoch {epoch+1}/{BERT_EPOCHS} — Loss: {avg_loss:.4f} ({elapsed:.0f}s elapsed)")

total_time = time.perf_counter() - t0
print(f"\nTraining complete in {total_time:.0f}s")

## 19. Inference — ModernBERT Predictions

In [ ]:
model.eval()
all_preds, all_labels, all_logits = [], [], []

with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        logits = model(**batch).logits
        all_preds.extend(torch.argmax(logits, dim=-1).cpu().numpy())
        all_labels.extend(batch["labels"].cpu().numpy())
        all_logits.append(logits.cpu())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_logits = torch.cat(all_logits, dim=0)
all_probs = torch.softmax(all_logits, dim=-1).numpy()

# ── Metrics ──────────────────────────────────────────────
bert_acc = accuracy_score(all_labels, all_preds)
bert_f1 = f1_score(all_labels, all_preds)
bert_prec = precision_score(all_labels, all_preds)
bert_rec = recall_score(all_labels, all_preds)
bert_auc = roc_auc_score(all_labels, all_probs[:, 1])

print(f"ModernBERT Results:  ({total_time:.0f}s training)")
print(f"  Accuracy:  {bert_acc:.4f}")
print(f"  Precision: {bert_prec:.4f}")
print(f"  Recall:    {bert_rec:.4f}")
print(f"  F1:        {bert_f1:.4f}")
print(f"  ROC-AUC:   {bert_auc:.4f}")
print()
print(classification_report(all_labels, all_preds,
      target_names=["Reliable", "Fake"], zero_division=0))

## 20. Evaluation — Model Comparison

### 20a. Confusion Matrix — ModernBERT

In [ ]:
cm_bert = confusion_matrix(all_labels, all_preds)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Baseline confusion matrix
sns.heatmap(cm_baseline, annot=True, fmt="d", cmap="Blues", ax=axes[0],
            xticklabels=["Reliable", "Fake"], yticklabels=["Reliable", "Fake"])
axes[0].set_title("Logistic Regression (TF-IDF)", fontsize=13)
axes[0].set_ylabel("Actual")
axes[0].set_xlabel("Predicted")

# ModernBERT confusion matrix
sns.heatmap(cm_bert, annot=True, fmt="d", cmap="Greens", ax=axes[1],
            xticklabels=["Reliable", "Fake"], yticklabels=["Reliable", "Fake"])
axes[1].set_title("ModernBERT", fontsize=13)
axes[1].set_ylabel("Actual")
axes[1].set_xlabel("Predicted")

plt.suptitle("Confusion Matrix Comparison", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

### 20b. ROC Curves

ROC curves show the tradeoff between true positive rate and false positive rate at different classification thresholds. Higher AUC = better discrimination.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

# Baseline ROC
fpr_bl, tpr_bl, _ = roc_curve(y_test, y_prob_baseline)
ax.plot(fpr_bl, tpr_bl, label=f"Logistic Regression (AUC={bl_auc:.4f})", linewidth=2)

# ModernBERT ROC
fpr_bert, tpr_bert, _ = roc_curve(all_labels, all_probs[:, 1])
ax.plot(fpr_bert, tpr_bert, label=f"ModernBERT (AUC={bert_auc:.4f})", linewidth=2)

# Random baseline
ax.plot([0, 1], [0, 1], "k--", alpha=0.4, label="Random (AUC=0.5)")

ax.set_xlabel("False Positive Rate", fontsize=12)
ax.set_ylabel("True Positive Rate", fontsize=12)
ax.set_title("ROC Curve Comparison", fontsize=14)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

### 20c. Summary Table

In [ ]:
comparison = pd.DataFrame([
    {"Model": "Logistic Regression (TF-IDF)", "Accuracy": round(bl_acc, 4),
     "Precision": round(bl_prec, 4), "Recall": round(bl_rec, 4),
     "F1": round(bl_f1, 4), "ROC-AUC": round(bl_auc, 4),
     "Training Time": f"{baseline_time:.1f}s"},
    {"Model": "ModernBERT", "Accuracy": round(bert_acc, 4),
     "Precision": round(bert_prec, 4), "Recall": round(bert_rec, 4),
     "F1": round(bert_f1, 4), "ROC-AUC": round(bert_auc, 4),
     "Training Time": f"{total_time:.0f}s"},
])
print(comparison.to_string(index=False))

## 21. Inference — Predict New Articles

Test both models on made-up news headlines/articles to see how they behave on unseen text.

In [ ]:
sample_articles = [
    "Scientists at MIT have developed a new vaccine that shows 95% efficacy in clinical trials involving 30,000 participants across 50 countries.",
    "EXPOSED: Secret government program uses 5G towers to control citizens' minds through microchip implants distributed via vaccines!!!",
    "The Federal Reserve announced a 0.25% interest rate increase, citing persistent inflation above the 2% target.",
    "SHOCKING: Hillary Clinton spotted meeting with aliens at Area 51, sources say she plans to hand over nuclear codes.",
    "Apple reported quarterly revenue of $94.8 billion, a 2% year-over-year increase driven by strong iPhone 15 sales.",
]

# ── TF-IDF + LogReg predictions ──────────────────────────
new_tfidf = tfidf.transform(sample_articles)
bl_preds = best_baseline.predict(new_tfidf)
bl_probs_new = best_baseline.predict_proba(new_tfidf)[:, 1]

# ── ModernBERT predictions ───────────────────────────────
model.eval()
with torch.no_grad():
    enc = tokenizer(sample_articles, truncation=True, padding="max_length",
                    max_length=BERT_MAX_LEN, return_tensors="pt").to(device)
    logits = model(**enc).logits
    bert_preds_new = torch.argmax(logits, dim=-1).cpu().numpy()
    bert_probs_new = torch.softmax(logits, dim=-1).cpu().numpy()[:, 1]

print("Predictions on new articles:\n")
for i, text in enumerate(sample_articles):
    preview = text[:120] + "..." if len(text) > 120 else text
    bl_label = "FAKE" if bl_preds[i] == 1 else "REAL"
    bert_label = "FAKE" if bert_preds_new[i] == 1 else "REAL"
    print(f'  "{preview}"')
    print(f"    LogReg: {bl_label} (prob={bl_probs_new[i]:.3f})")
    print(f"    BERT:   {bert_label} (prob={bert_probs_new[i]:.3f})")
    print()

## 22. Error Analysis

Understanding **why** the model fails is more valuable than the accuracy number. We look at misclassified examples from the ModernBERT model to identify patterns.

This reveals:
- Articles with ambiguous tone (reliable sources reporting sensational topics)
- Cases where writing style mimics the "wrong" class
- Potential labeling errors in the dataset

In [ ]:
# ── Collect misclassified examples from ModernBERT ────────
test_texts = X_test_text.tolist()
test_labels_arr = y_test.values

# False Positives: real articles predicted as fake
fp_mask = (all_preds == 1) & (all_labels == 0)
fp_indices = np.where(fp_mask)[0]

# False Negatives: fake articles predicted as real
fn_mask = (all_preds == 0) & (all_labels == 1)
fn_indices = np.where(fn_mask)[0]

print(f"False Positives (real → predicted fake): {fp_mask.sum()}")
print(f"False Negatives (fake → predicted real): {fn_mask.sum()}")

print(f"\n{'='*60}")
print("FALSE POSITIVES — Real articles mistakenly flagged as fake:")
print(f"{'='*60}")
for i, idx in enumerate(fp_indices[:3]):
    text = test_texts[idx]
    preview = text[:300] + "..." if len(text) > 300 else text
    conf = all_probs[idx, 1]
    print(f"\n[{i+1}] Confidence fake: {conf:.3f}")
    print(f"    {preview}")

print(f"\n{'='*60}")
print("FALSE NEGATIVES — Fake articles that slipped through:")
print(f"{'='*60}")
for i, idx in enumerate(fn_indices[:3]):
    text = test_texts[idx]
    preview = text[:300] + "..." if len(text) > 300 else text
    conf = all_probs[idx, 1]
    print(f"\n[{i+1}] Confidence fake: {conf:.3f}")
    print(f"    {preview}")

## 23. Limitations

1. **Dataset age:** The training data is from 2015–2018 US political news — it may not generalize to current events, non-US news, or non-political domains
2. **Binary oversimplification:** "Fake" is a spectrum — satire, propaganda, misleading framing, and outright fabrication are very different phenomena
3. **Style vs substance:** The model learns writing style patterns, not factual accuracy — a well-written lie may pass, and poorly written truth may be flagged
4. **No fact-checking:** The model has no access to ground truth or external knowledge — it cannot verify claims
5. **Label noise:** Some "fake" labels may be incorrect, and some "real" labels may be questionable
6. **Truncation:** ModernBERT truncates articles to 256 tokens (~200 words) — important context at the end of long articles is lost
7. **No temporal validation:** Random splits allow temporal leakage — the model may learn event-specific patterns that don't generalize

## 24. How to Improve This Project

| Improvement | Difficulty | Expected Impact |
|-------------|------------|-----------------|
| **Add title as a feature** | Easy | Title often reveals clickbait patterns |
| **Temporal train/test split** | Easy | More realistic performance estimate |
| **Increase ModernBERT max_len to 512** | Easy | Better for long articles |
| **Use title + text concatenation** | Easy | More input signal |
| **Class-weighted training** | Easy | Better if dataset is imbalanced |
| **Cross-domain evaluation** | Medium | Test on non-political news |
| **Ensemble (TF-IDF + transformer)** | Medium | Usually +1–2% accuracy |
| **Multi-class labels** (satire, propaganda, etc.) | Hard | More nuanced detection |
| **Fact-checking integration** (claim extraction + verification) | Hard | Actual factual accuracy checking |
| **Source credibility features** | Medium | Author/source reputation as features |

## 25. Production Considerations

- **Latency:** TF-IDF + LogReg predicts in <1ms per article. ModernBERT takes ~10–50ms on GPU.
- **Model size:** TF-IDF vectorizer + LogReg is ~50–200MB. ModernBERT is ~500MB.
- **Confidence threshold:** In production, flag low-confidence predictions for human review rather than auto-classifying.
- **Adversarial robustness:** Sophisticated fake news creators will adapt writing style to evade detection — models need regular retraining.
- **Ethical concerns:** False positives (flagging real news as fake) can be harmful to legitimate journalism. Err on the side of caution.
- **Monitoring:** Track prediction distribution shifts over time — concept drift is guaranteed as news topics change.
- **Human-in-the-loop:** Use the model as a screening tool, not a final arbiter. Human fact-checkers should make the final call.
- **Re-training cadence:** Retrain monthly or quarterly on fresh labeled data.
- **Legal compliance:** Ensure compliance with content moderation regulations in your jurisdiction.

## 26. Common Mistakes

| Mistake | Why It's Wrong | What to Do Instead |
|---------|---------------|-------------------|
| Using accuracy alone | On balanced data it's fine, but it masks precision/recall tradeoffs | Always report precision, recall, F1, and confusion matrix |
| Training on future data | Temporal leakage inflates metrics | Use temporal splits for realistic evaluation |
| Equating "style detection" with "fact checking" | The model detects writing patterns, not factual truth | Be honest about what the model actually does |
| Over-engineering preprocessing | Heavy regex cleaning or stemming often hurts | Let TF-IDF handle tokenization; keep preprocessing minimal |
| Deploying without a confidence threshold | Forces binary decisions on ambiguous cases | Set a threshold; route low-confidence cases to humans |
| Not checking for author leakage | Author name can be highly predictive due to dataset bias | Drop author features, or verify the model works without them |
| Fine-tuning a transformer when TF-IDF works fine | Wasted GPU time, higher latency, no meaningful gain | Start with baselines; upgrade only if the gap justifies cost |

## 27. Mini Challenge

Test your understanding with these exercises:

1. **Add the article title as a feature:** Concatenate `title + " " + text` before TF-IDF. Does accuracy improve? Why might it?

2. **Try different TF-IDF settings:** Set `ngram_range=(1, 3)` (include trigrams) and `max_features=50000`. Is the improvement worth the extra computation?

3. **Add class weights:** Use `LogisticRegression(class_weight="balanced")`. How does this change precision vs. recall?

4. **Increase ModernBERT max_len to 512:** Does accuracy improve? How much slower is training?

5. **Inspect the most confident wrong predictions:** Find the 5 articles where ModernBERT was most confident but wrong. What patterns do you see?

6. **Build a simple ensemble:** Average the LogReg probability with the ModernBERT probability. Does the ensemble beat both individual models?

## 28. Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | **TF-IDF + Logistic Regression is a strong baseline** — often 90%+ on binary text classification |
| 2 | **Fake news detection is style detection, not fact-checking** — the model learns writing patterns, not truth |
| 3 | **Confusion matrix > accuracy** — understanding false positives vs false negatives is critical |
| 4 | **ROC-AUC** measures discrimination ability independent of the classification threshold |
| 5 | **Error analysis** on actual misclassified examples teaches you more than any metric |
| 6 | **ModernBERT** can improve over TF-IDF, but the cost (time, GPU) must justify the gain |
| 7 | **Dataset limitations** (age, label quality, domain) constrain what any model can achieve |
| 8 | **Production fake news detection** requires human-in-the-loop, not autonomous classification |

---

*This notebook is part of the [Machine Learning Projects](https://github.com/pypi-ahmad/Machine-Learning-Projects) repository.*